# 🚗 Geo-AI Training — Workshop 4: นับรถเข้า-ออกจากวิดีโอโดรนด้วย Computer Vision

**แนวคิด**: ใช้โมเดล AI สำเร็จรูป (YOLOv8 pretrained) ตรวจจับรถในวิดีโอโดรน + ติดตามรถแต่ละคัน (tracking) แล้วนับว่าคันไหนวิ่งผ่าน "เส้นนับ" ที่ตั้งไว้ในทิศทางใด (เข้า/ออก) — แสดงผลแบบเห็นวิดีโอ+กรอบตรวจจับ**ระหว่างประมวลผลไปเรื่อยๆ เลย** ไม่ต้องรอ export เสร็จก่อนถึงจะดูผลได้

> ⚠️ ความเร็วในการประมวลผลขึ้นกับเครื่อง (โดยเฉพาะถ้าไม่มี GPU) อาจช้ากว่าความเร็ววิดีโอจริง — แต่จุดสำคัญคือเห็นผลลัพธ์ค่อยๆ ขึ้นทีละเฟรมสด ไม่ใช่รอผลสรุปตอนจบ

## 🧩 Setup

> 📦 ก่อนรัน ติดตั้ง dependencies ให้ครบ
> ```
> pip install -r requirements.txt
> ```
> ⚠️ ครั้งแรกที่รัน จะดาวน์โหลดโมเดล YOLOv8m (~50MB) อัตโนมัติจากอินเทอร์เน็ต

In [ ]:
# ติดตั้ง library ที่ใช้ในไฟล์นี้ (ultralytics จะโหลดโมเดล YOLOv8m ~50MB ให้อัตโนมัติตอนใช้งานครั้งแรก)
!pip install -q ultralytics gdown lab

print("ติดตั้งเสร็จแล้ว ✅")

### ⚠️ อย่าลืมเปิด GPU ก่อนรัน Workshop นี้

Workshop นี้ใช้ YOLOv8 ตรวจจับ+ติดตามรถทุกเฟรมของวิดีโอ ถ้ารันด้วย CPU จะช้ามาก

**วิธีเปิด GPU**: เมนู `Runtime` → `Change runtime type` → เลือก **Hardware accelerator = GPU** → กด Save (ถ้าเพิ่งเปลี่ยน ต้องรันทุกเซลล์ใหม่ตั้งแต่ต้น)

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"✅ พบ GPU: {torch.cuda.get_device_name(0)} — จะประมวลผลได้เร็วมาก")
else:
    print("⚠️ ไม่พบ GPU — กำลังใช้ CPU (จะช้ากว่ามาก) ไปที่ Runtime > Change runtime type > GPU แล้วรันใหม่")

### ติดตั้งฟอนต์ไทย (สำหรับกราฟที่มีข้อความไทย)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import urllib.request
import os

# โหลดฟอนต์ Sarabun (ฟอนต์ไทยจาก Google Fonts) มาใช้กับกราฟ — โหลดครั้งเดียว ถ้ามีไฟล์แล้วข้ามได้เลย
font_path = "Sarabun-Regular.ttf"
if not os.path.exists(font_path):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/google/fonts/main/ofl/sarabun/Sarabun-Regular.ttf",
        font_path
    )
fm.fontManager.addfont(font_path)
plt.rcParams["font.family"] = "Sarabun"
plt.rcParams["axes.unicode_minus"] = False

print("ติดตั้งฟอนต์ไทยเสร็จแล้ว ✅")

In [ ]:
import os

# 📁 ใช้พื้นที่เก็บไฟล์ชั่วคราวในเครื่อง Colab (ไม่เชื่อม Google Drive)
# ⚠️ ไฟล์ในนี้จะหายไปเมื่อ Colab runtime ถูกตัดการเชื่อมต่อ/รีสตาร์ท — ดาวน์โหลดเก็บเองก่อนปิดเครื่อง
# (ใช้แผง Files ด้านซ้ายของ Colab คลิกขวาไฟล์ > Download)
DATA_DIR = "/content/data"
OUTPUT_DIR = "/content/outputs"
EXPORT_DIR = os.path.join(OUTPUT_DIR, "export")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(EXPORT_DIR, exist_ok=True)

print("DATA_DIR   :", DATA_DIR)
print("EXPORT_DIR :", EXPORT_DIR)

### โหลดวิดีโอโดรนจาก Google Drive

วิดีโอโดรนถ่ายนิ่ง (hover) มองถนนจากด้านบน มีรถวิ่งทั้งสองทิศทาง

In [ ]:
import gdown

VIDEO_ID = "1N_-l5TpmsC117Z6QMZ3V_iutxHVtx7Y4"
video_path = os.path.join(DATA_DIR, "traffic.mp4")

if os.path.exists(video_path):
    print("✅ มีไฟล์อยู่แล้ว: traffic.mp4")
else:
    print("⬇️  กำลังโหลด: traffic.mp4 ...")
    gdown.download(id=VIDEO_ID, output=video_path, quiet=False)

print("เสร็จแล้ว พร้อมใช้งาน")

## 🔬 เตรียมวิดีโอ: ย่อขนาด + crop เฉพาะช่วงถนนตรง

วิดีโอโดรนความละเอียดสูงเกินความจำเป็น และมีส่วนที่ไม่เกี่ยวกับการนับรถ (ตึก/ต้นไม้สองข้างทาง, ทางแยกใกล้กล้องด้านล่างที่รถเลี้ยว/กลับรถทำให้นับพลาดง่าย) จึง crop เหลือเฉพาะช่วงถนนตรงตรงกลาง แล้วย่อขนาดลง ช่วยให้ตรวจจับเร็วขึ้นและแม่นยำขึ้น

In [ ]:
import cv2

cap = cv2.VideoCapture(video_path)
FRAME_W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
FRAME_H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
FPS = cap.get(cv2.CAP_PROP_FPS)
N_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

print(f"ขนาดวิดีโอต้นฉบับ: {FRAME_W} x {FRAME_H} พิกเซล")
print(f"FPS: {FPS:.1f}, จำนวนเฟรม: {N_FRAMES}, ความยาว: {N_FRAMES/FPS:.1f} วินาที")

In [ ]:
# 🔧 ปรับได้: สัดส่วนความกว้างตรงกลางที่จะ crop (0.45 = เก็บ 45% ตรงกลาง ตัดสองข้างทิ้ง)
CROP_FRAC_W = 0.45
# 🔧 ปรับได้: สัดส่วนความสูงจากขอบบนที่จะเก็บไว้ — 0.78 เก็บเกือบถึงทางแยก (เผื่อรถใกล้กล้อง เช่น รถตู้/รถกระบะ
# ไม่ให้โดน crop ตัดออกจนจับไม่ได้เลย) แต่ยังตัดจุดที่รถเลี้ยว/กลับรถจริงๆ ออกอยู่
CROP_FRAC_H = 0.78
# 🔧 ปรับได้: ความกว้างหลังย่อขนาด (ยิ่งน้อยยิ่งเร็ว แต่รถไกลๆ อาจเล็กเกินจับ)
TARGET_WIDTH = 960

crop_x0 = int(FRAME_W * (0.5 - CROP_FRAC_W / 2))
crop_x1 = int(FRAME_W * (0.5 + CROP_FRAC_W / 2))
crop_y1 = int(FRAME_H * CROP_FRAC_H)
crop_w = crop_x1 - crop_x0
scale = TARGET_WIDTH / crop_w
out_w, out_h = TARGET_WIDTH, int(crop_y1 * scale)

def prep_frame(frame):
    """Crop กลางเฟรม (ตัดทั้งขอบซ้าย-ขวา และช่วงล่างที่ไม่ใช่ถนนตรง) แล้วย่อขนาด"""
    cropped = frame[0:crop_y1, crop_x0:crop_x1]
    return cv2.resize(cropped, (out_w, out_h))

print(f"ขนาดหลัง crop+ย่อ: {out_w} x {out_h} พิกเซล")

In [ ]:
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(video_path)
ok, frame = cap.read()
cap.release()

preview = prep_frame(frame)
plt.figure(figsize=(5, 8))
plt.imshow(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB))
plt.title("ตัวอย่างเฟรมหลัง crop + ย่อขนาด")
plt.axis("off")
plt.show()

## 🔬 โหลดโมเดล AI (YOLOv8) และตั้งเส้นนับรถ

In [ ]:
from ultralytics import YOLO

# 🔧 ปรับได้: "n"(nano)/"s"(small)/"m"(medium) — ใหญ่ขึ้นแม่นยำขึ้นแต่ช้าลง
# ใช้ "m" เพราะรถบางประเภท (เช่น รถตู้ที่ถ่ายจากมุมโดรน) โมเดลเล็กมักตรวจจับพลาด
model = YOLO("yolov8m.pt")

# คลาสรถใน COCO dataset: 2=car, 3=motorcycle, 5=bus, 7=truck
VEHICLE_CLASSES = [2, 3, 5, 7]
# 🔧 ปรับได้: ค่าความมั่นใจขั้นต่ำที่จะนับว่าเป็นรถ — ลดลงถ้ายังจับรถไม่ครบ (แต่อาจได้ false positive เพิ่มขึ้น)
# หมายเหตุ: ลดต่ำกว่านี้มากๆ (เช่น 0.05) จะเจอ false positive จากป้าย/หลังคาตึกเยอะขึ้นมาก ไม่แนะนำ
CONF_THRESHOLD = 0.2

print("โหลดโมเดลเสร็จแล้ว ✅")

### ตั้งเส้นนับ (Counting Line)

ตั้งเส้นแนวนอน — รถที่ศูนย์กลาง (centroid) เคลื่อนผ่านเส้นนี้จะถูกนับ 1 ครั้ง โดยดูทิศทางจากการเคลื่อนที่ขึ้น/ลง

> ⚠️ **สมมติฐานของ workshop นี้**: รถที่เคลื่อน**ลง**ผ่านเส้น (เข้าใกล้กล้องมากขึ้น) = **"ขาเข้า"**, รถที่เคลื่อน**ขึ้น** (ห่างจากกล้อง) = **"ขาออก"** — ถ้าเปลี่ยนไปใช้วิดีโอมุมอื่น ต้องปรับทิศทางให้ตรงกับสถานการณ์จริงเอง
>
> 💡 ตั้งเส้นไว้ที่ตำแหน่งค่อนไปทางบน (ไม่ใช่กลางเฟรมพอดี) เพราะ crop เผื่อพื้นที่ด้านล่างไว้กว้างเพื่อจับรถใกล้กล้อง (เช่น รถตู้) — ถ้าเอาเส้นไปไว้ใกล้ทางแยกด้านล่างเกินไป รถที่เลี้ยว/ชะลอจะทำให้นับสับสนได้

In [ ]:
LINE_Y_FRAC = 0.37  # 🔧 ปรับได้: ตำแหน่งเส้นนับ (0=บนสุดของเฟรม, 1=ล่างสุด)
LINE_Y = int(out_h * LINE_Y_FRAC)

print(f"ตั้งเส้นนับที่ y = {LINE_Y} (จากความสูงเฟรม {out_h})")

## 🎬 ประมวลผลแบบ Real-time

แสดงผลทีละเฟรมสดๆ ระหว่างประมวลผล (ใช้เทคนิค `clear_output` + แสดงภาพซ้ำในเซลล์เดิม) แทนที่จะรอ export วิดีโอเสร็จก่อนค่อยดู

In [ ]:
FRAME_SKIP = 1     # 🔧 ปรับได้: ประมวลผลทุกๆ N เฟรม (เพิ่มขึ้นถ้าเครื่องช้า/วิดีโอยาว)
DISPLAY_EVERY = 2  # 🔧 ปรับได้: อัปเดตภาพที่แสดงทุกๆ N เฟรมที่ประมวลผล (ถี่เกินไปจะกระตุก)
PROCESS_LAST_SECONDS = 15  # 🔧 ปรับได้: ตัดวิดีโอเหลือแค่ N วินาทีสุดท้าย (None = ประมวลผลทั้งคลิป)

if PROCESS_LAST_SECONDS is not None:
    start_frame = max(0, N_FRAMES - int(PROCESS_LAST_SECONDS * FPS))
else:
    start_frame = 0

counts = {"ขาเข้า": 0, "ขาออก": 0}
prev_positions = {}  # track_id -> y-center ล่าสุด

print(f"เริ่มประมวลผลตั้งแต่เฟรม {start_frame} ถึง {N_FRAMES} (ทุกๆ {FRAME_SKIP} เฟรม)...")

In [ ]:
def process_frame(frame):
    """รับเฟรมดิบ 1 เฟรม -> ตรวจจับ+ติดตามรถ, อัปเดต counts/prev_positions, คืนภาพที่วาดกรอบ+เส้นนับแล้ว"""
    small = prep_frame(frame)
    results = model.track(small, persist=True, classes=VEHICLE_CLASSES, conf=CONF_THRESHOLD, verbose=False)
    annotated = results[0].plot()  # วาดกรอบ + track ID ให้อัตโนมัติ

    # เช็คว่ารถคันไหนวิ่งผ่านเส้นนับบ้าง (เทียบตำแหน่งเฟรมนี้กับเฟรมก่อนหน้า)
    boxes = results[0].boxes
    if boxes.id is not None:
        ids = boxes.id.cpu().numpy().astype(int)
        xyxy = boxes.xyxy.cpu().numpy()
        for tid, (x1, y1, x2, y2) in zip(ids, xyxy):
            cy = (y1 + y2) / 2
            if tid in prev_positions:
                prev_y = prev_positions[tid]
                if prev_y < LINE_Y <= cy:
                    counts["ขาเข้า"] += 1
                elif prev_y > LINE_Y >= cy:
                    counts["ขาออก"] += 1
            prev_positions[tid] = cy

    # วาดเส้นนับ + ตัวเลขสรุปทับบนภาพ (ใช้ IN/OUT เพราะ OpenCV วาดฟอนต์ไทยไม่ได้)
    cv2.line(annotated, (0, LINE_Y), (out_w, LINE_Y), (0, 0, 255), 2)
    cv2.putText(annotated, f"IN: {counts['ขาเข้า']}   OUT: {counts['ขาออก']}",
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
    return annotated

print("สร้างฟังก์ชัน process_frame เสร็จแล้ว ✅")

### ทดสอบกับ 1 เฟรมก่อน

ลองเรียก `process_frame` กับเฟรมเดียวดูก่อน เพื่อเช็คว่ากรอบตรวจจับ/เส้นนับถูกต้องก่อนรันทั้งคลิป

In [ ]:
cap = cv2.VideoCapture(video_path)
cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
ok, test_frame = cap.read()
cap.release()

test_annotated = process_frame(test_frame)

plt.figure(figsize=(6, 7.5))
plt.imshow(cv2.cvtColor(test_annotated, cv2.COLOR_BGR2RGB))
plt.title("ทดสอบตรวจจับ 1 เฟรม")
plt.axis("off")
plt.show()

### รีเซ็ตค่านับก่อนรันจริง

การทดสอบด้านบนไปแตะ `counts`/`prev_positions` ไว้แล้ว ต้องรีเซ็ตให้เป็น 0 ก่อนรันทั้งคลิปจริง

In [ ]:
counts = {"ขาเข้า": 0, "ขาออก": 0}
prev_positions = {}

print("รีเซ็ตค่านับเรียบร้อย ✅ พร้อมรันจริง")

### รันจริงทั้งคลิป แสดงผลสด

In [ ]:
from IPython.display import display, clear_output, Image as IPyImage

cap = cv2.VideoCapture(video_path)
cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)  # ข้ามไปเริ่มที่ช่วงวินาทีสุดท้ายตามที่ตั้งไว้
frame_i = start_frame

while True:
    ok, frame = cap.read()
    if not ok:
        break
    frame_i += 1
    if frame_i % FRAME_SKIP != 0:
        continue

    annotated = process_frame(frame)

    if frame_i % (FRAME_SKIP * DISPLAY_EVERY) == 0:
        _, buf = cv2.imencode(".jpg", annotated)
        clear_output(wait=True)
        display(IPyImage(data=buf.tobytes()))
        print(f"เฟรม {frame_i}/{N_FRAMES} | ขาเข้า: {counts['ขาเข้า']} | ขาออก: {counts['ขาออก']}")

cap.release()
clear_output(wait=True)
print("ประมวลผลเสร็จสมบูรณ์ ✅")
print(f"สรุป: ขาเข้า {counts['ขาเข้า']} คัน, ขาออก {counts['ขาออก']} คัน")

## 📊 สรุปผล

In [ ]:
plt.figure(figsize=(4, 4))
plt.bar(counts.keys(), counts.values(), color=["#2b8a3e", "#e03131"])
plt.title(f"จำนวนรถที่นับได้ (ทั้งหมด {sum(counts.values())} คัน)")
plt.ylabel("จำนวน (คัน)")
for i, v in enumerate(counts.values()):
    plt.text(i, v, str(v), ha="center", va="bottom")
plt.tight_layout()
plt.show()

## 💾 Export ผลลัพธ์

In [ ]:
import csv

csv_path = os.path.join(EXPORT_DIR, "traffic_counts.csv")
with open(csv_path, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(["direction", "count"])
    for direction, n in counts.items():
        writer.writerow([direction, n])

print("✅ Export แล้ว:", csv_path)

---
✅ **จบ Workshop 4** — ไปต่อที่ `5_export_summary.ipynb`